# Generar Tensor para ConvLSTM (Sit. 3)

1. Descarga tiles S2 y embeddings .npz desde HF
2. Lee panel S5P desde GCS (autenticacion Colab) o desde HF si esta disponible
3. **Carga embeddings directamente desde .npz** (generado por sit2_sae_posttrain.ipynb)
4. Genera secuencias + targets T+1/T+3/T+7
5. Construye tensor (N, 8, 522, 5, 5) y sube a HF

### -*Esto se transformo a un Script .py y se hizo la corrida para generar el tensor y subirlo a HF*

In [ ]:
# @title 0. Instalar dependencias
%pip install -q huggingface_hub zarr numcodecs pandas pyarrow numpy scipy
%pip install -q torch tqdm matplotlib xarray gcsfs

In [ ]:
from __future__ import annotations
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from huggingface_hub import snapshot_download, HfApi
from tqdm.notebook import tqdm

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


## 1. Descargar tiles S2 + embeddings .npz desde HF

El archivo `embeddings_sit2.npz` fue generado por `sit2_sae_posttrain.ipynb`
y contiene features visuales (2263, 512) listos para usar.
Ya NO es necesario cargar RemoteCLIP ni extraer features.

In [ ]:
# @title Descargar dataset_sit2
HF_TOKEN = os.environ.get("HF_TOKEN")
DATA_DIR = Path("/content/dataset_sit2")

if not (DATA_DIR / "metadatos.parquet").is_file():
    print("Descargando dataset_sit2...")
    snapshot_download(
        repo_id="Slucu-0310/geovision-cali-sit2",
        repo_type="dataset",
        local_dir=str(DATA_DIR),
        local_dir_use_symlinks=False,
        token=HF_TOKEN,
    )
print("Dataset OK")

In [ ]:
# @title Cargar metadatos + embeddings desde .npz
import zarr

# Cargar metadatos
df = pd.read_parquet(DATA_DIR / "metadatos.parquet")
df["fecha_dt"] = pd.to_datetime(df["fecha"])
print(f"Tiles: {len(df)} | fechas: {df['fecha_dt'].min()} a {df['fecha_dt'].max()}")

# Cargar embeddings desde .npz (generado por sit2_sae_posttrain.ipynb)
npz_path = DATA_DIR / "embeddings_sit2.npz"
if not npz_path.is_file():
    # Fallback: buscar en subdirectorios
    import glob
    cand = list(Path("/content").rglob("embeddings_sit2.npz"))
    if cand:
        npz_path = cand[0]
    else:
        raise FileNotFoundError(
            "No se encuentra embeddings_sit2.npz. Debe generarlo ejecutando "
            "sit2_sae_posttrain.ipynb (celda de exportacion .npz) y luego "
            "subirlo a HuggingFace o copiarlo a dataset_sit2/."
        )

data_npz = np.load(npz_path)
embeddings = data_npz["features"]  # (N, 512)
labels_npz = data_npz["labels"]
print(f"Embeddings .npz: {embeddings.shape}")
print(f"Clases disponibles: {np.unique(labels_npz)}")

# Verificar alineacion
assert len(embeddings) == len(df), (
    f"Desajuste: embeddings {len(embeddings)} vs parquet {len(df)}"
)
print("Alineacion embeddings-parquet: OK")
data_npz.close()

In [ ]:
# @title Cargar tiles Zarr (opcional, para indices NDVI/BSI)
store = zarr.storage.LocalStore(str(DATA_DIR / "tiles.zarr"))
tiles_z = zarr.open(store, mode="r")
if isinstance(tiles_z, zarr.Group):
    tiles_z = tiles_z["tiles"]
print(f"Tiles Zarr: {tiles_z.shape}")

## 2. Panel S5P (desde GCS via Colab o HF)

Carga los datos de Sentinel-5P (NO2, SO2, O3) para extraer los targets
de contaminacion en T+1, T+3 y T+7.

In [ ]:
# @title Autenticar GCP en Colab y cargar panel S5P
import xarray as xr
import gcsfs

GCS_BUCKET = "gs://geovision-cali"

# Autenticar GCP (funciona en Colab automaticamente)
from google.colab import auth
auth.authenticate_user()
print("Autenticado GCP")

# Conectar a GCS
fs = gcsfs.GCSFileSystem(project="proyecto3ia-494900")

# Cargar paneles S5P
s5p_panels = {}
for pol in ["NO2", "SO2", "O3"]:
    path = f"{GCS_BUCKET}/Sentinel5P/{pol}/panel.zarr"
    ds = xr.open_zarr(fs.get_mapper(path), consolidated=True)
    s5p_panels[pol] = ds
    print(f"S5P {pol}: {len(ds.time)} fechas, y={ds.dims["y"]}, x={ds.dims["x"]}")

## 3. Secuencias + targets T+1/T+3/T+7

Agrupa tiles por celda de 0.05deg, ordena por fecha, y genera ventanas
de 8 timesteps con targets de NO2, SO2, O3 en los horizontes especificados.

In [ ]:
# @title Generar secuencias y extraer targets S5P
RES = 0.05
SEQ_LEN = 8
HORIZONS = [1, 3, 7]

df["celda_lat"] = (df["centroide_lat"] / RES).round() * RES
df["celda_lon"] = (df["centroide_lon"] / RES).round() * RES
df["embedding"] = [embeddings[i] for i in range(len(embeddings))]

def get_s5p(panel, fecha, lat, lon, max_days=3):
    times = pd.to_datetime(panel["time"].values)
    target = np.datetime64(fecha)
    diffs = np.abs(times - target)
    idx = np.argmin(diffs)
    if diffs[idx] > np.timedelta64(max_days, "D"):
        return np.nan
    yi = np.argmin(np.abs(panel["y"].values - lat))
    xi = np.argmin(np.abs(panel["x"].values - lon))
    val = float(panel["data"].isel(time=idx, band=0, y=yi, x=xi).values)
    return val if np.isfinite(val) and val > 0 else np.nan

sequences = []
for (clat, clon), grp in df.groupby(["celda_lat", "celda_lon"]):
    ord_grp = grp.sort_values("fecha_dt")
    fechas_u = ord_grp["fecha_dt"].unique()
    if len(fechas_u) < SEQ_LEN:
        continue
    for i in range(len(fechas_u) - SEQ_LEN + 1):
        ventana = fechas_u[i:i+SEQ_LEN]
        rows = [ord_grp[ord_grp["fecha_dt"] == f].iloc[0] for f in ventana]
        last_date = ventana[-1]
        targets = []
        for h in HORIZONS:
            tgt = last_date + pd.Timedelta(days=h)
            targets.append([get_s5p(s5p_panels[p], tgt, clat, clon) for p in ["NO2","SO2","O3"]])
        sequences.append({
            "celda_lat": clat, "celda_lon": clon,
            "rows": rows,
            "targets": np.array(targets, dtype=np.float32),
        })
print(f"Secuencias: {len(sequences)}")

## 4. Construir tensor y subir a HF

C = 522 = 512 (embedding) + 4 (doy sin/cos, mes sin/cos) + 2 (lat, lon norm)
         + 1 (gap temporal) + 3 (NDVI, BSI, NDBI)
Dimension: (N, 8, 522, 5, 5)

In [ ]:
# @title Construir X=(N,8,522,5,5) + y=(N,3,3)
C, GS, STRIDE = 522, 5, 0.005

def build_X(rows):
    lat_c, lon_c = rows[0]["centroide_lat"], rows[0]["centroide_lon"]
    half = (GS - 1) / 2
    lats = np.linspace(lat_c - half*STRIDE, lat_c + half*STRIDE, GS)
    lons = np.linspace(lon_c - half*STRIDE, lon_c + half*STRIDE, GS)
    X = np.zeros((len(rows), C, GS, GS), dtype=np.float32)
    for t, row in enumerate(rows):
        emb, fecha = row["embedding"], row["fecha_dt"]
        doy = fecha.dayofyear
        for gi in range(GS):
            for gj in range(GS):
                X[t, :512, gi, gj] = emb
                X[t, 512, gi, gj] = np.sin(2*np.pi*doy/365.25)
                X[t, 513, gi, gj] = np.cos(2*np.pi*doy/365.25)
                X[t, 514, gi, gj] = np.sin(2*np.pi*fecha.month/12)
                X[t, 515, gi, gj] = np.cos(2*np.pi*fecha.month/12)
                X[t, 516, gi, gj] = (lats[gi] - 3.45) / 0.1
                X[t, 517, gi, gj] = (lons[gj] + 76.5) / 0.1
                X[t, 518, gi, gj] = 0.0 if t == 0 else min((rows[t]["fecha_dt"] - rows[t-1]["fecha_dt"]).days / 30.0, 5.0)
                X[t, 519, gi, gj] = row.get("ndvi", 0) if np.isfinite(row.get("ndvi", 0)) else 0
                X[t, 520, gi, gj] = row.get("bsi", 0) if np.isfinite(row.get("bsi", 0)) else 0
                X[t, 521, gi, gj] = row.get("ndbi", 0) if np.isfinite(row.get("ndbi", 0)) else 0
    return X

X_list, y_list = [], []
for seq in tqdm(sequences):
    X_list.append(build_X(seq["rows"]))
    y_list.append(seq["targets"])

X_full = np.stack(X_list, axis=0)
y_full = np.stack(y_list, axis=0)
print(f"X: {X_full.shape}, y: {y_full.shape}")

In [ ]:
# @title Guardar y subir a HF
np.save(DATA_DIR / "X_convlstm.npy", X_full)
np.save(DATA_DIR / "y_convlstm.npy", y_full)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)
SIT3_REPO = "Slucu-0310/geovision-cali-sit3"
api.create_repo(SIT3_REPO, repo_type="dataset", exist_ok=True)
api.upload_file(path_or_fileobj=str(DATA_DIR/"X_convlstm.npy"),
                path_in_repo="X_convlstm.npy", repo_id=SIT3_REPO, repo_type="dataset")
api.upload_file(path_or_fileobj=str(DATA_DIR/"y_convlstm.npy"),
                path_in_repo="y_convlstm.npy", repo_id=SIT3_REPO, repo_type="dataset")
print(f"Subido a https://huggingface.co/datasets/{SIT3_REPO}")

# Opcional: subir tambien el .npz usado
if npz_path.is_file():
    api.upload_file(
        path_or_fileobj=str(npz_path),
        path_in_repo="embeddings_sit2.npz",
        repo_id=SIT3_REPO, repo_type="dataset",
    )
    print("  embeddings_sit2.npz tambien subido")